# Explore Stockdata API

### Import Libraries

In [ ]:
from alpaca.data.requests import StockBarsRequest
from alpaca.data.requests import StockLatestQuoteRequest
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.timeframe import TimeFrame

from datetime import datetime
from pathlib import Path
import plotly.graph_objects as go
import pandas as pd
import math
import os

from dotenv import load_dotenv

ROOT      = Path.cwd()
DATA_DIR  = ROOT / "data"
CACHE_DIR = DATA_DIR / "cache"


### Load stockbars from Alpaca API

In [44]:
load_dotenv()
client = StockHistoricalDataClient(
    api_key=os.getenv("ALPACA_API_KEY"),
    secret_key=os.getenv("ALPACA_SECRET_KEY")
)

bars_request_params = StockBarsRequest(
    symbol_or_symbols=["AAPL"],
    timeframe=TimeFrame.Day,
    start=datetime(2020, 1, 1),
    end=datetime(2023, 12, 31),
    adjustment= "all"
)

multisymbol_request_params = StockLatestQuoteRequest(
    symbol_or_symbols=["SPY", "GLD", "TLT"]
)

bars = client.get_stock_bars(bars_request_params).df
print("Datatype of bars:", type(bars))
print("Index levels of bars:", bars.index.nlevels)
print("Columns of bars:", bars.columns.tolist())
bars.head()

Datatype of bars: <class 'pandas.DataFrame'>
Index levels of bars: 2
Columns of bars: ['open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']


open   high    low  close       volume  \
symbol timestamp                                                            
AAPL   2020-01-02 05:00:00+00:00  71.28  72.33  71.03  72.27  146217352.0   
       2020-01-03 05:00:00+00:00  71.50  72.33  71.35  71.57  154711256.0   
       2020-01-06 05:00:00+00:00  70.69  72.18  70.44  72.14  125599968.0   
       2020-01-07 05:00:00+00:00  72.15  72.40  71.58  71.80  118227136.0   
       2020-01-08 05:00:00+00:00  71.50  73.26  71.50  72.95  142736804.0   

                                  trade_count   vwap  
symbol timestamp                                      
AAPL   2020-01-02 05:00:00+00:00     283489.0  71.84  
       2020-01-03 05:00:00+00:00     280441.0  71.91  
       2020-01-06 05:00:00+00:00     242868.0  71.66  
       2020-01-07 05:00:00+00:00     227871.0  71.96  
       2020-01-08 05:00:00+00:00     261773.0  72.50

### Describe and check the data

In [45]:
print(bars.info())
print("NaN values in bars:\n", bars.isna().sum())
print("Unique index:", bars.index.is_unique)
print("Monotonic increasing:", bars.index.is_monotonic_increasing)

<class 'pandas.DataFrame'>
MultiIndex: 1006 entries, ('AAPL', Timestamp('2020-01-02 05:00:00+0000', tz='UTC')) to ('AAPL', Timestamp('2023-12-29 05:00:00+0000', tz='UTC'))
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         1006 non-null   float64
 1   high         1006 non-null   float64
 2   low          1006 non-null   float64
 3   close        1006 non-null   float64
 4   volume       1006 non-null   float64
 5   trade_count  1006 non-null   float64
 6   vwap         1006 non-null   float64
dtypes: float64(7)
memory usage: 65.9 KB
None
NaN values in bars:
 open           0
high           0
low            0
close          0
volume         0
trade_count    0
vwap           0
dtype: int64
Unique index: True
Monotonic increasing: True


### Plot stockbars data

In [46]:
bars_appl = bars.loc["AAPL"]
print("Apple bars info", bars_appl.info())

fig = go.Figure(
    data=[go.Candlestick(
        x=bars_appl.index,
        open=bars_appl['open'],
        high=bars_appl['high'],
        low=bars_appl['low'],
        close=bars_appl['close']
    )],
    layout=go.Layout(
        title='AAPL Stock Price',
        xaxis_title='Date',
        yaxis_title='Price (USD)',
        xaxis_rangeslider_visible=False
    )
)
fig.show()

<class 'pandas.DataFrame'>
DatetimeIndex: 1006 entries, 2020-01-02 05:00:00+00:00 to 2023-12-29 05:00:00+00:00
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   open         1006 non-null   float64
 1   high         1006 non-null   float64
 2   low          1006 non-null   float64
 3   close        1006 non-null   float64
 4   volume       1006 non-null   float64
 5   trade_count  1006 non-null   float64
 6   vwap         1006 non-null   float64
dtypes: float64(7)
memory usage: 62.9 KB
Apple bars info None


### Calculte revenue for different timeframes

In [47]:
def revenue(start_date, end_date, stock_df):
    
    start_ts = pd.to_datetime(start_date).tz_localize("UTC")
    end_ts = pd.to_datetime(end_date).tz_localize("UTC")

    # look for the first available price on or after the start date
    buy_price = stock_df.loc[stock_df.index >= start_ts, "close"].iloc[0]

    # look for the last available price on or before the end date
    sell_price = stock_df.loc[stock_df.index <= end_ts, "close"].iloc[-1]
    
    return (sell_price / buy_price) - 1

In [48]:
roi_apple_2020_2023 = revenue("2021-01-01", "2022-10-31", bars_appl)
print(f"Revenue for AAPL from 2021-01-01 to 2022-10-31: {roi_apple_2020_2023:.2%}")

Revenue for AAPL from 2021-01-01 to 2022-10-31: 21.60%


### Calculate the Capital Deployment

One way to calculate the capital deployment is by multiplying the start capital with the revenue for the given timeframe. 

The formula is as follows:
$$
V_t=V_0\prod_{i=1}^{t}(1+r_i)
$$

In [49]:
def cap_depl(start_capital, timestamps, stock_df):
    return start_capital * math.prod(
      1 + revenue(buy, sell, stock_df) for (buy, sell) in timestamps
  )

In [50]:
# test the cap_depl function with a sample start capital and timestamps
start_capital = 10000
timestamps = [("2021-01-01", "2021-06-30"),
              ("2021-07-01", "2021-12-31"),
              ("2022-01-01", "2022-06-30"),
              ("2022-07-01", "2022-12-31")]
capital_deployment = cap_depl(start_capital, timestamps, bars_appl)
print(f"Capital deployment for AAPL from 2021-01-01 to 2022-12-31 with start capital ${start_capital}: ${capital_deployment:.2f}")

Capital deployment for AAPL from 2021-01-01 to 2022-12-31 with start capital $10000: $9900.22


In [51]:
# Buy and hold strategy for AAPL from 2021-01-01 to 2022-12-31 with a start capital of $10,000
start_capital = 10000
timestamps = [("2021-01-01", "2022-12-31")]
capital_deployment = cap_depl(start_capital, timestamps, bars_appl)
print(f"Capital deployment for AAPL from 2021-01-01 to 2022-12-31 with start capital ${start_capital}: ${capital_deployment:.2f}")


Capital deployment for AAPL from 2021-01-01 to 2022-12-31 with start capital $10000: $10160.79


An other way to calculate the capital deployment is to use the start capital and the stock price at the start and end of the timeframe. 
$$
V_t=\underbrace{\frac{V_0}{P_0}}_{\text{Number of Stocks}}\cdot P_t
=V_0\cdot\frac{P_t}{P_0}
$$

In [52]:
start_ts = pd.to_datetime("2021-01-01").tz_localize("UTC")
end_ts = pd.to_datetime("2022-12-31").tz_localize("UTC")

p_0 = bars_appl.loc[bars_appl.index >= start_ts, "close"].iloc[0]
p_t = bars_appl.loc[bars_appl.index <= end_ts, "close"].iloc[-1]
v0 = start_capital
v_t = v0 * (p_t / p_0)
print(f"Capital deployment for AAPL from 2021-01-01 to 2022-12-31 with a start capital of ${v0}: ${v_t:.2f}")

Capital deployment for AAPL from 2021-01-01 to 2022-12-31 with a start capital of $10000: $10160.79


Both methods should yield the same result for capital deployment over a given timeframe.

This results in the follwing assumption of the market:

- Fractional shares allowed
- No fees and taxes
- No separately considered dividends
- No deposits or withdrawals of capital
- Cash earns no interest, meaning that any cash not invested in the stock does not grow over time.